# Switch Variable Identifier

**Goal:** Build a tool that identifies, ranks, and measures the impact of switch variables using the Claude API.

**What you'll do:**
1. Use Claude to extract candidate switch variables from a domain + question
2. Ask the same question with and without each variable
3. Measure how much each variable shifts the response
4. Rank variables by their measured information gain

**Time:** ~15 minutes

**Prerequisite:** `ANTHROPIC_API_KEY` set in your environment, `pip install anthropic`

## Setup

In [ ]:
import os
import json
import re
import anthropic
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Verify API key is available
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY not set. Run: export ANTHROPIC_API_KEY='your-key-here'"
    )

client = anthropic.Anthropic()
print("Client ready.")

## Step 1: Extract Switch Variable Candidates

Given a domain and a question, we ask Claude to identify the conditions that would change the correct answer entirely.

The prompt uses the five-category framework from Guide 1: jurisdiction, timing, status, constraints, objective function.

In [ ]:
def extract_switch_variables(domain: str, question: str) -> list[dict]:
    """
    Ask Claude to identify switch variables for a domain + question.

    Returns a list of dicts, each with:
      - name: short label (e.g. "jurisdiction")
      - category: one of the five categories
      - value_a: first representative value
      - value_b: second representative value (the contrast)
      - rationale: why this variable changes the answer
    """
    prompt = f"""Domain: {domain}
Question: {question}

Identify the top 5 switch variables for this question — conditions that would route
reasoning to a CATEGORICALLY different answer, not just a refined version of the same answer.

For each switch variable, provide:
- name: a short label (3-5 words)
- category: one of [jurisdiction, timing, status, constraints, objective]
- value_a: a representative first value
- value_b: a representative contrasting value
- rationale: one sentence explaining why this variable changes the answer category

Respond ONLY with a JSON array. No prose before or after.

Example format:
[
  {{
    "name": "governing jurisdiction",
    "category": "jurisdiction",
    "value_a": "US federal",
    "value_b": "UK",
    "rationale": "US and UK employment law have fundamentally different termination rules."
  }}
]"""

    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.content[0].text.strip()

    # Strip markdown code fence if present
    raw = re.sub(r"^```[\w]*\n?", "", raw)
    raw = re.sub(r"\n?```$", "", raw)

    return json.loads(raw)


# --- Run it ---
DOMAIN = "US employment law"
QUESTION = "Can I terminate this employee without severance?"

switch_vars = extract_switch_variables(DOMAIN, QUESTION)

print(f"Switch variables identified for: '{QUESTION}'")
print(f"Domain: {DOMAIN}\n")
for i, sv in enumerate(switch_vars, 1):
    print(f"{i}. [{sv['category'].upper()}] {sv['name']}")
    print(f"   Values: '{sv['value_a']}' vs '{sv['value_b']}'")
    print(f"   Why: {sv['rationale']}")
    print()

## Step 2: Measure Impact — Same Question With and Without Each Variable

For each switch variable, we run the question in two conditions:
- **Baseline:** no conditions specified
- **With value A:** condition added with its first value
- **With value B:** condition added with its contrasting value

We then measure how much the responses diverge. Larger divergence = higher information gain.

The similarity metric is a simple word-overlap score (Jaccard similarity). A categorical change in answer typically produces low overlap.

In [ ]:
def ask_with_condition(question: str, condition_name: str, condition_value: str) -> str:
    """Ask the question with a single condition specified."""
    conditioned_question = f"{condition_name}: {condition_value}\n\nQuestion: {question}"
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=400,
        messages=[{"role": "user", "content": conditioned_question}],
    )
    return response.content[0].text.strip()


def ask_baseline(question: str) -> str:
    """Ask the question with no conditions at all."""
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=400,
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text.strip()


def jaccard_similarity(text_a: str, text_b: str) -> float:
    """
    Word-overlap similarity between two responses.
    Returns 0.0 (completely different) to 1.0 (identical word sets).
    """
    words_a = set(text_a.lower().split())
    words_b = set(text_b.lower().split())
    if not words_a or not words_b:
        return 0.0
    return len(words_a & words_b) / len(words_a | words_b)


def measure_impact(sv: dict, question: str, baseline: str) -> dict:
    """
    For one switch variable, get responses for value_a and value_b.
    Measure how different they are from the baseline and from each other.
    """
    resp_a = ask_with_condition(question, sv["name"], sv["value_a"])
    resp_b = ask_with_condition(question, sv["name"], sv["value_b"])

    sim_a_baseline = jaccard_similarity(resp_a, baseline)
    sim_b_baseline = jaccard_similarity(resp_b, baseline)
    sim_a_b = jaccard_similarity(resp_a, resp_b)

    # Impact score: how different are the two conditioned responses from each other?
    # High divergence between value_a and value_b = high information gain
    impact = 1.0 - sim_a_b

    return {
        "name": sv["name"],
        "category": sv["category"],
        "value_a": sv["value_a"],
        "value_b": sv["value_b"],
        "response_a": resp_a,
        "response_b": resp_b,
        "similarity_a_b": sim_a_b,
        "similarity_a_baseline": sim_a_baseline,
        "similarity_b_baseline": sim_b_baseline,
        "impact_score": impact,
    }


# --- Run baseline first ---
print("Running baseline (no conditions)...")
baseline_response = ask_baseline(QUESTION)
print(f"Baseline response preview: {baseline_response[:200]}...\n")

In [ ]:
# --- Measure impact for each switch variable ---
# Note: this makes 10 API calls (2 per variable × 5 variables)
print("Measuring impact for each switch variable...")
print("(2 API calls per variable)\n")

results = []
for sv in switch_vars:
    print(f"Testing: {sv['name']}...")
    result = measure_impact(sv, QUESTION, baseline_response)
    results.append(result)
    print(f"  Impact score: {result['impact_score']:.3f} (A-B similarity: {result['similarity_a_b']:.3f})")

# Sort by impact score descending
results.sort(key=lambda x: x["impact_score"], reverse=True)

print("\nRanked by information gain:")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['name']} (impact: {r['impact_score']:.3f})")

## Step 3: Visualize the Rankings

The impact score is `1 - Jaccard(response_A, response_B)`. Higher impact = the two values of this variable produced more different responses = higher information gain.

In [ ]:
# Category colors
CATEGORY_COLORS = {
    "jurisdiction": "#2563eb",  # blue
    "timing": "#16a34a",         # green
    "status": "#9333ea",          # purple
    "constraints": "#ea580c",     # orange
    "objective": "#dc2626",       # red
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Impact scores (ranked) ---
names = [r["name"] for r in results]
scores = [r["impact_score"] for r in results]
colors = [CATEGORY_COLORS.get(r["category"], "#6b7280") for r in results]

bars = ax1.barh(names, scores, color=colors, edgecolor="white", linewidth=0.5)
ax1.set_xlabel("Impact Score (1 - Jaccard similarity between value A and B responses)")
ax1.set_title(f"Switch Variable Impact Ranking\n'{QUESTION}'")
ax1.set_xlim(0, 1.0)

for bar, score in zip(bars, scores):
    ax1.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{score:.3f}",
        va="center",
        fontsize=9,
    )

legend_patches = [
    mpatches.Patch(color=color, label=cat)
    for cat, color in CATEGORY_COLORS.items()
]
ax1.legend(handles=legend_patches, loc="lower right", fontsize=8)

# --- Plot 2: Response similarity matrix ---
# Compare each variable's value_a and value_b responses to the baseline
# Low similarity to baseline = high divergence from unconditioned answer

n = len(results)
x = np.arange(n)
width = 0.35

sim_a = [r["similarity_a_baseline"] for r in results]
sim_b = [r["similarity_b_baseline"] for r in results]

ax2.bar(x - width / 2, sim_a, width, label="Value A vs baseline", alpha=0.8, color="#3b82f6")
ax2.bar(x + width / 2, sim_b, width, label="Value B vs baseline", alpha=0.8, color="#f97316")
ax2.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, label="Identical to baseline")
ax2.set_xlabel("Switch Variable")
ax2.set_ylabel("Similarity to Baseline Response")
ax2.set_title("How Much Each Condition Shifts the Response")
ax2.set_xticks(x)
ax2.set_xticklabels(
    [r["name"].split()[0] for r in results],
    rotation=20,
    ha="right",
    fontsize=9,
)
ax2.set_ylim(0, 1.15)
ax2.legend(fontsize=8)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("Low similarity to baseline = the condition moved the answer significantly.")
print("Low similarity between A and B = the variable is a true switch (categorical change).")

## Step 4: Inspect the Highest-Gain Variable

See the actual responses side by side for the highest-ranked switch variable.

In [ ]:
top = results[0]

print("=" * 70)
print(f"HIGHEST-GAIN SWITCH VARIABLE: {top['name'].upper()}")
print(f"Category: {top['category']}")
print(f"Impact score: {top['impact_score']:.3f}")
print("=" * 70)

print(f"\n--- Baseline (no conditions) ---")
print(baseline_response)

print(f"\n--- With: {top['name']} = '{top['value_a']}' ---")
print(top["response_a"])

print(f"\n--- With: {top['name']} = '{top['value_b']}' ---")
print(top["response_b"])

print("\n" + "=" * 70)
print(f"Similarity between A and B responses: {top['similarity_a_b']:.3f}")
print("A score near 0 = categorically different answers (true switch variable)")
print("A score near 1 = same answer, just rephrased (low information gain)")

## Step 5: Build the Optimal Prompt

Use the ranked switch variables to construct a high-information-gain prompt. We take the top 3 by impact score and inject them as structured conditions.

In [ ]:
def build_conditioned_prompt(
    question: str,
    ranked_variables: list[dict],
    top_n: int = 3,
    chosen_values: dict | None = None,
) -> str:
    """
    Build a prompt using the top N switch variables.

    chosen_values: dict mapping variable name -> specific value to use.
    If not provided, uses value_a for each variable.
    """
    if chosen_values is None:
        chosen_values = {}

    conditions = []
    for sv in ranked_variables[:top_n]:
        value = chosen_values.get(sv["name"], sv["value_a"])
        conditions.append(f"- {sv['name'].title()}: {value}")

    conditions_block = "\n".join(conditions)
    return f"Conditions:\n{conditions_block}\n\nQuestion: {question}"


# Build a prompt using top 3 switch variables
optimal_prompt = build_conditioned_prompt(QUESTION, results, top_n=3)
print("Optimal prompt (top 3 switch variables):")
print("=" * 50)
print(optimal_prompt)
print("=" * 50)

# Get the response
print("\nResponse:")
optimal_response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=600,
    messages=[{"role": "user", "content": optimal_prompt}],
).content[0].text
print(optimal_response)

## Step 6: Compare Baseline vs. Optimal

Side-by-side view of what the model returns before and after high-gain conditioning.

In [ ]:
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6))

def format_text_for_axes(text: str, max_chars: int = 70) -> str:
    """Wrap long text for display in matplotlib axes."""
    import textwrap
    wrapped = textwrap.fill(text, width=max_chars)
    return wrapped


for ax, title, content, color in [
    (ax_left, "Baseline (no conditions)", baseline_response, "#fef2f2"),
    (ax_right, f"Top 3 switch variables specified", optimal_response, "#f0fdf4"),
]:
    ax.set_facecolor(color)
    ax.axis("off")
    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)
    ax.text(
        0.02, 0.97,
        format_text_for_axes(content[:800] + ("..." if len(content) > 800 else "")),
        transform=ax.transAxes,
        fontsize=8,
        verticalalignment="top",
        wrap=True,
        family="monospace",
    )

final_similarity = jaccard_similarity(baseline_response, optimal_response)
fig.suptitle(
    f"Question: '{QUESTION}'\n"
    f"Baseline vs. Conditioned Response — Similarity: {final_similarity:.3f} "
    f"({'low = categorical change' if final_similarity < 0.4 else 'high = similar answer'})",
    fontsize=10,
    y=1.02,
)
plt.tight_layout()
plt.show()

print(f"Similarity score: {final_similarity:.3f}")
print("Lower = the conditions changed the answer more fundamentally.")

## Try It Yourself

Change the domain and question below. The tool will identify switch variables, rank them by measured information gain, and build the optimal conditioned prompt for your case.

In [ ]:
# MODIFY THESE — try your own domain and question
MY_DOMAIN = "software architecture"
MY_QUESTION = "Should I use a microservices or monolithic architecture?"

# Optional: specify values you want used for the top switch variables
# Leave empty {} to use the first value Claude identifies for each
MY_CHOSEN_VALUES = {}

# --- Run the full pipeline ---
print(f"Domain: {MY_DOMAIN}")
print(f"Question: {MY_QUESTION}\n")

my_switches = extract_switch_variables(MY_DOMAIN, MY_QUESTION)
print("Identified switch variables:")
for i, sv in enumerate(my_switches, 1):
    print(f"  {i}. [{sv['category']}] {sv['name']}: '{sv['value_a']}' vs '{sv['value_b']}'")

print("\nMeasuring impact (this makes API calls for each variable)...")
my_baseline = ask_baseline(MY_QUESTION)

my_results = []
for sv in my_switches:
    r = measure_impact(sv, MY_QUESTION, my_baseline)
    my_results.append(r)
    print(f"  {sv['name']}: impact = {r['impact_score']:.3f}")

my_results.sort(key=lambda x: x["impact_score"], reverse=True)

print("\nRanked switch variables:")
for i, r in enumerate(my_results, 1):
    print(f"  {i}. {r['name']} (impact: {r['impact_score']:.3f}, category: {r['category']})")

print("\nOptimal prompt (top 3):")
my_prompt = build_conditioned_prompt(MY_QUESTION, my_results, top_n=3, chosen_values=MY_CHOSEN_VALUES)
print(my_prompt)

## Summary

**What we built:**

1. **Switch variable extractor** — uses Claude to identify the conditions that branch the answer space, organized by the five categories: jurisdiction, timing, status, constraints, objective

2. **Impact measurement** — runs the same question with each variable set to two contrasting values, measures how different the responses are (Jaccard divergence as a proxy for information gain)

3. **Ranked ranking** — sorts switch variables by their measured impact, confirming which conditions actually matter for this specific question

4. **Prompt builder** — constructs the optimal conditioned prompt using the top N ranked variables

**Key observation:** The top-ranked switch variable almost always corresponds to one of the five categories — usually jurisdiction, status, or objective function. Low-gain conditions (company background, formatting preferences, narrative context) never appear in the top 3.

**Next:** Exercise 1 — practice ranking switch variables for 10 real-world prompts across different domains, using the framework you just built.